<a href="https://colab.research.google.com/github/murilofarias10/MF-CCTB/blob/main/Association_Rules_single_itemsets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files

In [2]:
import pandas as pd

In [3]:
uploaded = files.upload()

Saving Catalog.xls to Catalog.xls


In [4]:
uploaded.keys()

dict_keys(['Catalog.xls'])

In [7]:
file_name = 'Catalog.xls'

In [8]:
df = pd.read_excel(file_name)
df.head()

,Customer_Number,Division
0,11569,Housewares Division
1,11569,Health Products Division
2,11569,Automotive Division
3,11569,Personal Electronics Division
4,11569,Novelty Gift Division


In [15]:
df = df.copy()
df.columns = [c.strip() for c in df.columns]

In [17]:
# Cell 4: Basic cleaning + keep required columns
df = df.copy()
df.columns = [c.strip() for c in df.columns]

required = {'Customer_Number', 'Division'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}. Found columns: {list(df.columns)}")

In [19]:

# Drop empty items
df = df.dropna(subset=['Customer_Number', 'Division'])
df['Division'] = df['Division'].astype(str).str.strip()
df = df[df['Division'] != '']

df.head(), df.shape

(   Customer_Number                       Division
 0            11569            Housewares Division
 1            11569       Health Products Division
 2            11569            Automotive Division
 3            11569  Personal Electronics Division
 4            11569          Novelty Gift Division,
 (14655, 2))

In [20]:
df.head(), df.shape

(   Customer_Number                       Division
 0            11569            Housewares Division
 1            11569       Health Products Division
 2            11569            Automotive Division
 3            11569  Personal Electronics Division
 4            11569          Novelty Gift Division,
 (14655, 2))

In [21]:
transactions = df.groupby('Customer_Number')['Division'].apply(list).tolist()
transactions[:3]

[['Housewares Division',
  'Health Products Division',
  'Automotive Division',
  'Personal Electronics Division',
  'Novelty Gift Division'],
 ['Housewares Division',
  'Health Products Division',
  'Automotive Division',
  'Personal Electronics Division',
  'Garden Division',
  'Novelty Gift Division',
  'Jewelry Division'],
 ['Housewares Division',
  'Health Products Division',
  'Automotive Division',
  'Personal Electronics Division',
  'Garden Division',
  'Novelty Gift Division',
  'Jewelry Division']]

In [22]:
from mlxtend.preprocessing import TransactionEncoder

In [23]:
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
basket = pd.DataFrame(te_ary, columns=te.columns_)
basket.head()

,Automotive Division,Clothing Division,Computers Division,Garden Division,Health Products Division,Housewares Division,Jewelry Division,Novelty Gift Division,Personal Electronics Division
0,True,False,False,False,True,True,False,True,True
1,True,False,False,True,True,True,True,True,True
2,True,False,False,True,True,True,True,True,True
3,True,False,False,True,True,False,False,True,True
4,False,False,False,True,True,False,False,True,True


In [24]:
# Cell 7: Calculate SUPPORT via frequent itemsets (Apriori)
from mlxtend.frequent_patterns import apriori

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [25]:
min_support = 0.01
itemsets = apriori(basket, min_support=min_support, use_colnames=True)
itemsets = itemsets.sort_values('support', ascending=False)
itemsets.head(10)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,support,itemsets
4,1.000000,(Health Products Division)
8,0.467387,(Personal Electronics Division)
35,0.467387,"(Health Products Division, Personal Electronic..."
32,0.393557,"(Housewares Division, Health Products Division)"
5,0.393557,(Housewares Division)
6,0.356943,(Jewelry Division)
33,0.356943,"(Jewelry Division, Health Products Division)"
27,0.272109,"(Health Products Division, Garden Division)"
3,0.272109,(Garden Division)
38,0.235494,"(Housewares Division, Personal Electronics Div..."


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Cell 8: Calculate CONFIDENCE via association rules
from mlxtend.frequent_patterns import association_rules

In [ ]:
min_confidence = 0.20
rules = association_rules(itemsets, metric='confidence', min_threshold=min_confidence)

In [ ]:


# Filter for single-item antecedents and consequents
rules_single = rules[
    (rules['antecedents'].apply(lambda x: len(x) == 1)) &
    (rules['consequents'].apply(lambda x: len(x) == 1))
]

# Select and sort
rules_conf = rules_single[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values('confidence', ascending=False)


rules_conf.head(10)

In [ ]:
rules_lift = rules_single[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values('lift', ascending=False)
rules_lift.head(10)

In [ ]:
def fs_to_str(x):
    return ', '.join(sorted(list(x)))

pretty = rules_lift.copy()
pretty['antecedents'] = pretty['antecedents'].apply(fs_to_str)
pretty['consequents'] = pretty['consequents'].apply(fs_to_str)
pretty.head(15)

In [ ]:
pretty.to_excel("output_file.xlsx", index=False)

In [38]:
files.download("output_file.xlsx")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag